# Phase 1 – Baseline Model (Logistic Regression)
This notebook establishes the baseline IPO success prediction model.
No further experiments should be added here.


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path


In [5]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_PROCESSED.mkdir(exist_ok=True)


In [6]:
ipo = pd.read_csv(DATA_RAW / "Initial Public Offering.csv")


In [7]:
ipo.head()


,Date,IPO_Name,Issue_Size(crores),QIB,HNI,RII,Total,Offer Price,List Price,Listing Gain,...,CMP(NSE),Current Gains,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,08/06/25,M & B Engineering Ltd,650,36.72,38.24,32.55,36.20,385,386,0.26,...,426.15,10.87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,08/06/25,Sri Lotus Developers & Realty Ltd,792,163.90,57.71,20.28,69.14,150,179.1,19.40,...,199.72,34.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,08/06/25,National Securities Depository Ltd (NSDL),"4,011.60",103.97,34.98,7.73,41.01,800,880,10.00,...,61.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,08/05/25,Aditya Infotech Ltd,"1,300",133.21,72.00,50.87,100.69,675,"1,018.00",50.81,...,"1,062.70",57.72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,08/05/25,Laxmi India Finance Ltd,254.26,1.30,1.84,2.22,1.87,158,136,-13.92,...,150,-5.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
ipo = ipo.rename(columns={
    "IPO_Name": "company_name",
    "Date": "listing_date",
    "Offer Price": "offer_price",
    "List Price": "list_price",
    "Listing Gain": "listing_gain"
})


In [9]:
ipo.columns


Index(['listing_date', 'company_name', 'Issue_Size(crores)', 'QIB', 'HNI',
       'RII', 'Total', 'offer_price', 'list_price', 'listing_gain', 'CMP(BSE)',
       'CMP(NSE)', 'Current Gains', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20'],
      dtype='object')

In [10]:
ipo = ipo.loc[:, ~ipo.columns.str.contains("^Unnamed")]


In [11]:
ipo.columns


Index(['listing_date', 'company_name', 'Issue_Size(crores)', 'QIB', 'HNI',
       'RII', 'Total', 'offer_price', 'list_price', 'listing_gain', 'CMP(BSE)',
       'CMP(NSE)', 'Current Gains'],
      dtype='object')

In [12]:
ipo = ipo.rename(columns={
    "listing_date": "listing_date",
    "company_name": "company",
    "Issue_Size(crores)": "issue_size_crore",
    "offer_price": "issue_price",
    "list_price": "listing_price",
    "listing_gain": "listing_gain_pct",
    "Current Gains": "current_gain_pct"
})


In [13]:
ipo["listing_date"] = pd.to_datetime(ipo["listing_date"], errors="coerce")

numeric_cols = [
    "issue_size_crore", "QIB", "HNI", "RII", "Total",
    "issue_price", "listing_price", "listing_gain_pct",
    "CMP(BSE)", "CMP(NSE)", "current_gain_pct"
]

for col in numeric_cols:
    ipo[col] = pd.to_numeric(ipo[col], errors="coerce")


/var/folders/fk/0kqynd_95d71j1htw9b057j00000gn/T/ipykernel_4089/1028216819.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ipo["listing_date"] = pd.to_datetime(ipo["listing_date"], errors="coerce")


In [14]:
ipo["listing_date"].head()


0   2025-08-06
1   2025-08-06
2   2025-08-06
3   2025-08-05
4   2025-08-05
Name: listing_date, dtype: datetime64[ns]

In [15]:
ipo["listing_date"].dtype


dtype('<M8[ns]')

In [16]:
ipo["success"] = (ipo["listing_gain_pct"] > 0).astype(int)


In [17]:
ipo.columns


Index(['listing_date', 'company', 'issue_size_crore', 'QIB', 'HNI', 'RII',
       'Total', 'issue_price', 'listing_price', 'listing_gain_pct', 'CMP(BSE)',
       'CMP(NSE)', 'current_gain_pct', 'success'],
      dtype='object')

In [18]:
ipo.columns = (
    ipo.columns
       .str.strip()
       .str.lower()
       .str.replace(" ", "_")
       .str.replace("(", "")
       .str.replace(")", "")
)


In [19]:
ipo.columns


Index(['listing_date', 'company', 'issue_size_crore', 'qib', 'hni', 'rii',
       'total', 'issue_price', 'listing_price', 'listing_gain_pct', 'cmpbse',
       'cmpnse', 'current_gain_pct', 'success'],
      dtype='object')

In [20]:
ipo["success"].value_counts()


success
1    389
0    172
Name: count, dtype: int64

In [21]:
ipo["success"].value_counts(normalize=True)


success
1    0.693405
0    0.306595
Name: proportion, dtype: float64

In [22]:
ipo.isna().mean().sort_values(ascending=False)


cmpnse              0.197861
cmpbse              0.181818
listing_price       0.089127
issue_price         0.048128
current_gain_pct    0.023173
issue_size_crore    0.003565
qib                 0.003565
hni                 0.003565
rii                 0.003565
total               0.003565
listing_date        0.000000
company             0.000000
listing_gain_pct    0.000000
success             0.000000
dtype: float64

In [23]:
ipo.describe()

,listing_date,issue_size_crore,qib,hni,rii,total,issue_price,listing_price,listing_gain_pct,cmpbse,cmpnse,current_gain_pct,success
count,561,559.000000,559.000000,559.000000,559.000000,559.000000,534.000000,511.000000,561.000000,459.000000,450.000000,548.000000,561.000000
mean,2019-02-28 23:00:57.754010880,1407.032021,47.477013,74.686834,15.972165,37.568891,313.316479,329.075851,18.029554,291.149717,295.597644,56.217431,0.693405
min,2010-01-04 00:00:00,23.000000,0.000000,0.000000,0.000000,0.110000,10.000000,9.500000,-31.730000,0.040000,0.050000,-99.930000,0.000000
25%,2015-05-26 00:00:00,251.140000,2.350000,1.710000,1.600000,2.245000,120.000000,130.050000,-0.010000,85.910000,91.307500,-36.365000,0.000000
50%,2021-02-05 00:00:00,588.220000,13.720000,13.200000,5.120000,11.520000,245.000000,264.000000,7.150000,221.100000,228.855000,13.960000,1.000000
75%,2023-11-07 00:00:00,1224.860000,76.255000,81.110000,13.750000,55.070000,447.500000,470.500000,26.130000,436.975000,445.762500,102.982500,1.000000
max,2025-08-06 00:00:00,27858.800000,331.600000,958.070000,374.810000,326.490000,985.000000,991.000000,252.760000,995.000000,994.900000,897.590000,1.000000
std,NaN,2644.873498,63.325282,136.785773,33.155961,53.804193,238.644552,241.232270,31.924815,258.902573,258.659058,152.844293,0.461491


In [24]:
FEATURES = [
    "issue_size_crore",
    "qib",
    "hni",
    "rii",
    "total",
    "issue_price"
]

TARGET = "success"


In [25]:
X = ipo[FEATURES]
y = ipo[TARGET]


In [26]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [27]:
feature_cols = [
    "issue_size_crore",
    "qib",
    "hni",
    "rii",
    "total",
    "issue_price"
]

X = ipo[feature_cols]
y = ipo["success"]


In [28]:
X.isna().sum()


issue_size_crore     2
qib                  2
hni                  2
rii                  2
total                2
issue_price         27
dtype: int64

In [29]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [31]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [32]:
y_pred = model.predict(X_test)


In [33]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6991150442477876
[[34  1]
 [33 45]]
              precision    recall  f1-score   support

           0       0.51      0.97      0.67        35
           1       0.98      0.58      0.73        78

    accuracy                           0.70       113
   macro avg       0.74      0.77      0.70       113
weighted avg       0.83      0.70      0.71       113



In [34]:
import pandas as pd

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coef_df


,Feature,Coefficient
4,total,0.024120
1,qib,0.019618
2,hni,0.005494
3,rii,0.005242
0,issue_size_crore,0.000047
5,issue_price,-0.000371


In [35]:
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)


In [36]:
import joblib

joblib.dump(model, "models/logistic_regression_baseline.pkl")


['models/logistic_regression_baseline.pkl']

In [37]:
MODELS_DIR.iterdir()


In [38]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]   # same logic as now
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# create folder if missing
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# IMPORTANT: df must be your final cleaned dataframe
ipo.to_csv(DATA_PROCESSED / "ipo_model_ready.csv", index=False)

print("Saved to:", DATA_PROCESSED / "ipo_model_ready.csv")


Saved to: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/data/processed/ipo_model_ready.csv


In [39]:
DATA_PROCESSED.exists(), list(DATA_PROCESSED.glob("*"))


(True,
 [PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/data/processed/ipo_model_ready.csv')])